# 33_Video 모델의 학습 수행하기

## 학습목표 
- 1. 커스텀 데이터셋 세팅 -> 학습의 순서로 딥러닝 학습 사이클을 만듭니다.   
- 2. GPU를 사용한 학습에서 주의할 점을 살펴봅니다.

In [1]:
import os
import cv2
import numpy as np
from collections import defaultdict

from torch.utils.data import Dataset
from torchvision import transforms

### 커스텀 데이터셋 세팅

In [2]:
# 데이터셋 루트 디렉토리 (사용자의 데이터 경로에 맞게 수정)
dataset_path = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_2/' # 실제 데이터 경로 입력

# UCF50 데이터셋 내 폴더(클래스) 목록 가져오기
classes = sorted([cls for cls in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, cls))])
num_classes = len(classes)

In [29]:
classes

['BaseballPitch',
 'Biking',
 'Diving',
 'Fencing',
 'HorseRace',
 'JumpRope',
 'PlayingGuitar',
 'PlayingPiano',
 'Punch',
 'SalsaSpin',
 'Skiing',
 'Swing',
 'TennisSwing',
 'VolleyballSpiking',
 'YoYo']

In [30]:
class CustomUCF50Dataset(Dataset):
    def __init__(self, root_dir, transform=None, num_frames=16):
        """
        UCF50 비디오 데이터셋을 PyTorch 데이터셋 클래스로 변환.
        
        :param root_dir: UCF50 데이터셋의 루트 디렉토리
        :param transform: 데이터 전처리 및 Augmentation
        :param num_frames: 샘플당 사용할 프레임 개수
        """
        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames

        # 클래스별 디렉토리를 탐색하여 비디오 파일을 리스트에 저장
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        print(f"CLS : {self.class_to_idx}")

        # 모든 비디오 파일의 경로 및 레이블을 리스트에 저장
        self.video_list = []
        for cls in self.classes:
            class_path = os.path.join(root_dir, cls)
            video_files = [f for f in os.listdir(class_path) if f.endswith(('.avi', '.mp4', '.mov', '.mkv'))]

            #print(video_files)
            for video in video_files:
                self.video_list.append((os.path.join(class_path, video), self.class_to_idx[cls]))

        #'C:/Users/jeong/Desktop/OnePM/Projects/Datasets/UCF50/UCF50/BaseballPitch\\v_BaseballPitch_g01_c01.avi'
        #print(self.video_list)

    def __len__(self):
        """ 데이터셋 크기 반환 """
        return len(self.video_list)

    def __getitem__(self, idx):
        """
        비디오 데이터를 읽어 PyTorch Tensor로 변환하여 반환.
        :param idx: 데이터 인덱스
        :return: (프레임 텐서, 레이블)
        """
        video_path, label = self.video_list[idx]

        # 비디오에서 프레임 로드
        frames = self._load_video_frames(video_path, self.num_frames)

        # 변환 적용 (torchvision.transforms 활용)
        if self.transform:
            frames = torch.stack([self.transform(frame) for frame in frames])

        return frames, torch.tensor(label, dtype=torch.long)

    def _load_video_frames(self, video_path, num_frames):
        """
        주어진 비디오에서 num_frames 개의 프레임을 균등한 간격으로 샘플링하여 반환.
        :param video_path: 비디오 파일 경로
        :param num_frames: 가져올 프레임 개수
        :return: [num_frames, H, W, C] 형태의 NumPy 배열 리스트
        """
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames == 0:
            cap.release()
            raise ValueError(f"비디오 {video_path}에서 프레임을 로드할 수 없습니다.")

        # 균등한 간격으로 프레임 샘플링
        frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
        frames = []

        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # OpenCV는 BGR 형식이므로 RGB로 변환
            frame = torch.tensor(frame, dtype=torch.float32) / 255.0  # [H, W, C] 정규화
            frames.append(frame)

        cap.release()

        # 프레임이 부족할 경우 마지막 프레임을 반복하여 채움
        while len(frames) < num_frames:
            frames.append(frames[-1].clone())

        frames = torch.stack(frames, dim=0)  # (num_frames, H, W, C)
        frames = frames.permute(0, 3, 1, 2)  # (num_frames, C, H, W)로 변환

        return frames

In [31]:
# 데이터 변환 설정 (Resizing + ToTensor)
transform = transforms.Compose([
    transforms.Resize((112, 112)),  # 입력 크기 맞추기
    #transforms.ToTensor(),
])

In [32]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 데이터셋 로드
dataset = CustomUCF50Dataset(root_dir=dataset_path, transform=transform, num_frames=5)
train_dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

CLS : {'BaseballPitch': 0, 'Biking': 1, 'Diving': 2, 'Fencing': 3, 'HorseRace': 4, 'JumpRope': 5, 'PlayingGuitar': 6, 'PlayingPiano': 7, 'Punch': 8, 'SalsaSpin': 9, 'Skiing': 10, 'Swing': 11, 'TennisSwing': 12, 'VolleyballSpiking': 13, 'YoYo': 14}


### 모델 준비

In [12]:
import torch
import torch.nn as nn
import torchvision.models as models

In [13]:
# ✅ MobileNetV3 백본 추출
mobilenet_v3 = models.mobilenet_v3_large(pretrained=True)
mobilenet_v3_backbone = mobilenet_v3.features  # 백본만 사용

class MobileNetFeatureExtractor(nn.Module):
    """ MobileNetV3에서 Feature Map을 추출하여 LSTM 입력 형태로 변환 """
    def __init__(self, backbone, output_dim=512):
        super().__init__()
        self.backbone = backbone  # ✅ MobileNetV3 CNN 백본
        self.global_pool = nn.AdaptiveAvgPool2d(1)  # ✅ Feature Map을 1x1로 축소
        self.fc = nn.Linear(960, output_dim)  # ✅ MobileNetV3 출력 채널(960)을 LSTM 입력 크기(512)로 변환

    def forward(self, x):
        batch_size, num_frames, channels, height, width = x.shape
        x = x.view(batch_size * num_frames, channels, height, width)  # [B*T, C, H, W]

        # ✅ CNN Backbone을 통해 Feature Map 추출
        features = self.backbone(x)  # [B*T, 960, H', W']
        features = self.global_pool(features)  # [B*T, 960, 1, 1]
        features = features.view(batch_size * num_frames, -1)  # [B*T, 960]

        # ✅ FC Layer를 통해 LSTM 입력 크기로 변환
        features = self.fc(features)  # [B*T, 512]
        features = features.view(batch_size, num_frames, -1)  # [B, T, 512]

        return features  # LSTM 입력 형식으로 변환된 Feature Vector

class ActionClassifier(nn.Module):
    def __init__(self, feature_extractor, num_classes, hidden_dim=256, num_layers=2):
        super().__init__()
        self.feature_extractor = feature_extractor  # ✅ MobileNetV3 기반 Feature Extractor
        self.lstm = nn.LSTM(input_size=512, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)  # ✅ CNN 백본으로 Feature 추출 → [B, T, 512]

        # ✅ LSTM으로 Temporal 정보 학습
        lstm_out, _ = self.lstm(features)  # [B, T, hidden_dim]
        action_logits = self.fc(lstm_out[:, -1, :])  # 마지막 타임스텝의 출력만 사용

        return action_logits  # [B, num_classes]

# ✅ 모델 생성
feature_extractor = MobileNetFeatureExtractor(mobilenet_v3_backbone)
action_classifier = ActionClassifier(feature_extractor, num_classes=15)

# ✅ 더미 입력 생성 (batch_size=2, num_frames=16, channels=3, height=224, width=224)
dummy_input = torch.randn(2, 16, 3, 224, 224)

# ✅ 예측 실행
output = action_classifier(dummy_input)
print("Output Shape:", output.shape)  # 예상 결과: [2, 10] (배치 크기, 클래스 수)

Output Shape: torch.Size([2, 15])


### 학습

In [25]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # 모델을 훈련 모드로 설정
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)  # GPU로 전송

        optimizer.zero_grad()  # 기존 그래디언트 초기화
        outputs = model(images)  # 모델 예측
        print(f"output : {outputs}")

        # UCF50은 Classification 문제 → outputs[-2]가 heatmap을 의미
        predictions = outputs
        predictions = predictions.view(predictions.size(0), -1)  # 펼치기
        loss = criterion(predictions, labels)  # 손실 계산

        loss.backward()  # 역전파
        optimizer.step()  # 옵티마이저 업데이트

        #print(f"할당된 메모리: {torch.cuda.memory_allocated() / 1024 ** 2:.2f} MB")
        #print(f"예약된 메모리: {torch.cuda.memory_reserved() / 1024 ** 2:.2f} MB")
        
        # 손실 및 정확도 계산
        total_loss += loss.item()
        total += labels.size(0)
        correct += (predictions.argmax(dim=1) == labels).sum().item()

        del images, labels, outputs  # 혹은 사용한 텐서들
        gc.collect()
        torch.cuda.empty_cache()

        # WandB 로깅
        #wandb.log({"Batch Loss": loss.item()})

        # 진행 상황 출력
        if batch_idx % 10 == 0:
            print(f"[Batch {batch_idx}/{len(dataloader)}] Loss: {loss.item():.4f}")

    # 에포크 손실 및 정확도 계산
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total

    # 최종 WandB 로깅
    #wandb.log({"Epoch Loss": avg_loss, "Accuracy": accuracy})
    
    print(f"\n[Epoch Finished] Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%\n")
    return avg_loss, accuracy

In [15]:
#device=0
print(torch.cuda.device_count())  # 사용 가능한 GPU 개수
print(torch.cuda.current_device())  # 현재 기본 GPU 번호
print(torch.cuda.get_device_name(torch.cuda.current_device()))  # 현재 사용 GPU 이름

1
0
NVIDIA GeForce RTX 4060 Laptop GPU


In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")

사용 중인 디바이스: cuda:0


In [17]:
import torch

device = torch.device("cuda:0")
print(torch.cuda.get_device_name(device))
print(f"총 메모리: {torch.cuda.get_device_properties(device).total_memory / 1024**2:.2f} GB")

NVIDIA GeForce RTX 4060 Laptop GPU
총 메모리: 8187.50 GB


In [18]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [19]:
print(f"총 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**2:.2f} MB")

총 메모리: 8187.50 MB


In [20]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim

# 3. 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(action_classifier.parameters(), lr=1e-4)

In [26]:
action_classifier.to(device)

# 5. 에포크 반복 훈련
num_epochs = 10
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    avg_loss, accuracy = train_one_epoch(action_classifier, train_dataloader, criterion, optimizer, device)

Epoch 1/10
output : tensor([[-0.0152, -0.0344, -0.0282,  0.0139, -0.0138, -0.0200,  0.0055, -0.0273,
         -0.0034,  0.0359,  0.0364,  0.0152,  0.0266, -0.0202,  0.0540],
        [-0.0019, -0.0463, -0.0497,  0.0097,  0.0123, -0.0188,  0.0107, -0.0134,
         -0.0070,  0.0458,  0.0681,  0.0174,  0.0340, -0.0384,  0.0462]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
[Batch 0/75] Loss: 2.7295
output : tensor([[-0.0125, -0.0600, -0.0071,  0.0556, -0.0303,  0.0026, -0.0106, -0.0425,
         -0.0051,  0.0257,  0.0316,  0.0002,  0.0269, -0.0332,  0.0697],
        [-0.0210, -0.0304, -0.0324,  0.0270, -0.0030, -0.0160,  0.0250, -0.0172,
         -0.0069,  0.0406,  0.0467,  0.0036,  0.0289, -0.0292,  0.0616]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
output : tensor([[-0.0167, -0.0544, -0.0236,  0.0620, -0.0141,  0.0008, -0.0086, -0.0274,
         -0.0156,  0.0084,  0.0347, -0.0184,  0.0281, -0.0171,  0.0968],
        [-0.0299, -0.0539, -0.0137,  0.0298,  0.0062,  0.0006, -

In [28]:
# 훈련 완료 후 가중치 저장
torch.save(action_classifier.state_dict(), "./video_model.pth")